In [ ]:
# @title 🛠️ 1. Cài đặt và Kết nối Google Drive
import warnings
warnings.filterwarnings('ignore')
from google.colab import drive
import os
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
!pip install -q curl_cffi ipywidgets pillow requests
print("✅ Hoàn tất cài đặt!")

In [ ]:
# @title ⚙️ 2. Hệ thống Tải Ảnh 100% TRỰC TIẾP PINTEREST WEB PINS
import warnings
warnings.filterwarnings('ignore')
import os, sys, time, json, hashlib, re, urllib.parse
from pathlib import Path
import requests
from PIL import Image
from curl_cffi import requests as cffi_requests

TARGET_W, TARGET_H = 1080, 1920

def search_pinterest_direct(query, limit=50):
    query_clean = urllib.parse.quote(query)
    search_url = f"https://www.pinterest.com/search/pins/?q={query_clean}"
    
    session = cffi_requests.Session()
    urls = []
    bookmarks = []
    
    try:
        r1 = session.get(search_url, impersonate="chrome124")
        csrf_token = session.cookies.get("csrftoken") or "123456"
        
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
            "Accept": "application/json, text/javascript, */*, q=0.01",
            "X-Requested-With": "XMLHttpRequest",
            "X-CSRFToken": csrf_token,
            "X-Pinterest-AppState": "active",
            "X-Pinterest-PWS-Handler": "www/search/pins.js",
            "Referer": search_url,
        }
        
        for page in range(5):
            options = {
                "isPrefetch": False,
                "query": query,
                "scope": "pins",
                "no_fetch_context_on_resource": False
            }
            if bookmarks:
                options["bookmarks"] = bookmarks
                
            data_payload = {
                "options": options,
                "context": {}
            }
            
            params = {
                "source_url": f"/search/pins/?q={query_clean}",
                "data": json.dumps(data_payload),
                "_": str(int(time.time() * 1000))
            }
            
            api_url = "https://www.pinterest.com/resource/BaseSearchResource/get/"
            r2 = session.get(api_url, params=params, headers=headers, impersonate="chrome124")
            
            if r2.status_code == 200:
                res_resp = r2.json().get("resource_response", {})
                results = res_resp.get("data", {}).get("results", [])
                new_b = res_resp.get("bookmark")
                if new_b:
                    bookmarks = [new_b]
                    
                for pin in results:
                    images = pin.get("images", {})
                    orig = images.get("orig", {}).get("url") or images.get("736x", {}).get("url") or images.get("474x", {}).get("url")
                    if orig and orig not in urls:
                        urls.append(orig)
                        
                if len(urls) >= limit or not new_b:
                    break
            else:
                break
                
            time.sleep(1)
    except Exception as e:
        print(f"Lỗi kết nối Pinterest Web Direct: {e}")
        
    seen = set()
    unique = []
    for u in urls:
        if u not in seen:
            seen.add(u)
            unique.append(u)
    return unique[:limit]

def resize_crop_save(media_data, out_path):
    tmp = out_path.parent / f"_tmp_{out_path.name}"
    tmp.write_bytes(media_data)
    try:
        img = Image.open(tmp).convert('RGB')
        w, h = img.size
        ratio = TARGET_W / TARGET_H
        if w/h > ratio:
            nh, nw = TARGET_H, int(w * (TARGET_H / h))
        else:
            nw, nh = TARGET_W, int(h * (TARGET_W / w))
        img = img.resize((nw, nh), Image.LANCZOS)
        l = (nw - TARGET_W) // 2
        t = (nh - TARGET_H) // 2
        img.crop((l, t, l + TARGET_W, t + TARGET_H)).save(out_path, 'JPEG', quality=92)
        tmp.unlink(missing_ok=True)
        return True
    except Exception:
        tmp.unlink(missing_ok=True)
        return False

def build_library(char_key, anime_name, base_dir, target=50):
    char_dir = base_dir / char_key
    char_dir.mkdir(parents=True, exist_ok=True)

    existing = list(char_dir.glob("*.jpg")) + list(char_dir.glob("*.png")) + list(char_dir.glob("*.jpeg")) + list(char_dir.glob("*.webp"))
    if len(existing) >= target:
        print(f"  ✅ [{char_key}]: Đã có đủ {len(existing)}/{target} ảnh Pinterest! (Bỏ qua không tải trùng)")
        return
    
    used_hashes = set()
    for f in existing:
        try:
            used_hashes.add(hashlib.md5(f.read_bytes()).hexdigest())
        except:
            pass

    query = f"{char_key.replace('_', ' ')} {anime_name.replace('_', ' ')}"
    print(f"🔎 Đang cào 100% Pinterest Web Pins cho '{query}' (Hiện có: {len(existing)}/{target})...")
    urls = search_pinterest_direct(query, limit=target * 2)
    
    saved_count = len(existing)
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
    for url in urls:
        if saved_count >= target:
            break
        try:
            r = requests.get(url, headers=headers, timeout=10)
            if r.status_code != 200 or len(r.content) < 8000:
                continue
            h = hashlib.md5(r.content).hexdigest()
            if h in used_hashes:
                continue
            used_hashes.add(h)
            
            out_file = char_dir / f"{char_key}_{saved_count+1:02d}.jpg"
            if resize_crop_save(r.content, out_file):
                saved_count += 1
                print(f"    + [{char_key}] #{saved_count:02d}: Đã lưu Pinterest Pin!")
        except Exception:
            continue
    print(f"  🎉 HOÀN THÀNH [{char_key}]: {saved_count}/{target} ảnh Pinterest!")

def run_fetch(anime_name):
    config_path = Path("/content/drive/MyDrive/anime_library/anime_characters_config.json")
    if not config_path.exists():
        print("LỖI: Chưa có file cấu hình anime_characters_config.json trên Drive!")
        return
    try:
        config = json.loads(config_path.read_text(encoding="utf-8"))
    except Exception as e:
        print(f"LỖI đọc config: {e}")
        return

    if anime_name not in config:
        print(f"LỖI: '{anime_name}' không có trong config! Các anime sẵn có: {list(config.keys())}")
        return
    
    char_dict = config[anime_name]
    base_dir = Path(f"/content/drive/MyDrive/anime_library/{anime_name}")
    base_dir.mkdir(parents=True, exist_ok=True)
    
    print(f"\n{'='*50}\n🚀 TẢI 100% PINTEREST PINS TRỰC TIẾP TỪ PINTEREST WEB: {anime_name}\n{'='*50}")
    for char_key in char_dict.keys():
        build_library(char_key, anime_name, base_dir, target=50)

In [ ]:
# @title 🎨 3. Giao diện Quản lý & Tải Ảnh (Chọn Anime -> Bấm nút tải)
import ipywidgets as widgets
from IPython.display import display, clear_output
import json
from pathlib import Path

CONFIG_PATH = Path('/content/drive/MyDrive/anime_library/anime_characters_config.json')

DEFAULT_DATA = {
    "Tensei_Slime": {
        "Rimuru_Tempest": ["Rimuru Tempest"],
        "Milim_Nava": ["Milim Nava"],
        "Benimaru": ["Benimaru"],
        "Veldora_Tempest": ["Veldora Tempest"]
    }
}
config_data = {}

def load_config():
    global config_data
    if CONFIG_PATH.exists():
        try:
            with open(CONFIG_PATH, 'r', encoding='utf-8') as f:
                config_data = json.load(f)
        except:
            config_data = DEFAULT_DATA.copy()
    else:
        config_data = DEFAULT_DATA.copy()
        CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)
        with open(CONFIG_PATH, 'w', encoding='utf-8') as f:
            json.dump(config_data, f, indent=4, ensure_ascii=False)

load_config()

out = widgets.Output()

anime_dropdown = widgets.Dropdown(options=list(config_data.keys()), description='Anime:', layout=widgets.Layout(width='300px'))
new_anime_input = widgets.Text(placeholder='Thêm Tên Anime Mới', layout=widgets.Layout(width='200px'))
add_anime_btn = widgets.Button(description='Thêm Anime', button_style='success')
del_anime_btn = widgets.Button(description='Xóa Anime', button_style='danger')

char_name_input = widgets.Text(placeholder='Tên Nhân Vật', layout=widgets.Layout(width='200px'))
add_char_btn = widgets.Button(description='Thêm NV', button_style='info')

run_fetch_btn = widgets.Button(description='🚀 TẢI ẢNH PINTEREST CHO ANIME ĐANG CHỌN', button_style='primary', layout=widgets.Layout(width='100%', height='50px'))

def update_ui(*args):
    sel_anime = anime_dropdown.value
    if sel_anime and sel_anime in config_data:
        chars = ", ".join(config_data[sel_anime].keys())
        with out:
            clear_output()
            print(f"📌 Nhân vật hiện tại trong '{sel_anime}': {chars}")

anime_dropdown.observe(update_ui, 'value')

def on_add_anime(b):
    nv = new_anime_input.value.strip().replace(" ", "_")
    if nv and nv not in config_data:
        config_data[nv] = {}
        anime_dropdown.options = list(config_data.keys())
        anime_dropdown.value = nv
        new_anime_input.value = ''
        save_conf()
        update_ui()

def on_del_anime(b):
    sel = anime_dropdown.value
    if sel in config_data:
        del config_data[sel]
        anime_dropdown.options = list(config_data.keys())
        save_conf()
        update_ui()

def on_add_char(b):
    sel_anime = anime_dropdown.value
    cname = char_name_input.value.strip().replace(" ", "_")
    if sel_anime and cname:
        config_data[sel_anime][cname] = [cname.replace("_", " ")]
        char_name_input.value = ''
        save_conf()
        update_ui()

def save_conf():
    with open(CONFIG_PATH, 'w', encoding='utf-8') as f:
        json.dump(config_data, f, indent=4, ensure_ascii=False)

def on_run_fetch(b):
    with out:
        clear_output()
        sel_anime = anime_dropdown.value
        print(f"⏳ Đang tải ảnh từ Pinterest cho Anime: {sel_anime}...")
        run_fetch(sel_anime)

add_anime_btn.on_click(on_add_anime)
del_anime_btn.on_click(on_del_anime)
add_char_btn.on_click(on_add_char)
run_fetch_btn.on_click(on_run_fetch)

if config_data: update_ui()

ui = widgets.VBox([
    widgets.HTML("<h2>⚙️ QUẢN LÝ THƯ VIỆN ẢNH ANIME TỪ PINTEREST</h2>"),
    widgets.HBox([anime_dropdown, new_anime_input, add_anime_btn, del_anime_btn]),
    widgets.HBox([widgets.Label("Thêm Nhân Vật Mới:"), char_name_input, add_char_btn]),
    run_fetch_btn,
    out
])
display(ui)

In [ ]:
# @title 🚀 4. Chạy Tải Ảnh Cho Anime Đang Chọn
sel_anime = anime_dropdown.value if 'anime_dropdown' in globals() else "Tensei_Slime"
run_fetch(sel_anime)